In [ ]:
from google.colab import driveimport osdrive.mount('/content/drive')DRIVE_BASE = '/content/drive/MyDrive/LoRA_Compress'os.makedirs(DRIVE_BASE, exist_ok=True)os.makedirs(f'{DRIVE_BASE}/databases', exist_ok=True)os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)

In [ ]:
%cd /content!git clone https://github.com/qades/loracompress.git 2>/dev/null || true%cd loracompress!pip install -q transformers torch optuna tqdm bitsandbytes accelerate

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fimport optunaimport randomimport numpy as npfrom transformers import AutoModelForCausalLMprint(f'PyTorch: {torch.__version__}')print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'print('Loading FP16 model...')model_fp16 = AutoModelForCausalLM.from_pretrained(    MODEL_NAME, torch_dtype=torch.float32, device_map='cpu', trust_remote_code=True)print('FP16 model loaded')# NF4 levelsNF4_LEVELS = torch.tensor([    -1.0, -0.696, -0.525, -0.395, -0.284, -0.185, -0.091, 0.0,    0.080, 0.161, 0.246, 0.338, 0.441, 0.563, 0.723, 1.0])def quantize_nf4(weight):    w = weight.float().flatten()    absmax = w.abs().max().clamp(min=1e-8)    w_norm = torch.clamp(w / absmax, -1.0, 1.0)    distances = (w_norm.unsqueeze(-1) - NF4_LEVELS).abs()    indices = distances.argmin(dim=-1)    dequant = NF4_LEVELS[indices] * absmax    return indices.reshape(weight.shape), dequant.reshape(weight.shape), absmaxprint('Quantization functions defined')

In [ ]:
print('Layer stats verification')TARGET_LAYERS = [    'model.layers.0.self_attn.q_proj',    'model.layers.15.self_attn.q_proj',    'model.layers.15.mlp.up_proj']layer_stats = []for target in TARGET_LAYERS:    for name, param in model_fp16.named_parameters():        if target in name and name.endswith('.weight'):            print(f'\n{name} shape: {tuple(param.shape)}')            w_fp16 = param.data            print(f'  FP16: mean={w_fp16.mean():.4f}, std={w_fp16.std():.4f}')            indices, dequant, absmax = quantize_nf4(w_fp16)            unique = len(torch.unique(indices))            print(f'  Q4: unique_indices={unique}/16, absmax={absmax:.4f}')            l1 = torch.mean(torch.abs(dequant - w_fp16)).item() / torch.mean(torch.abs(w_fp16)).item() * 100            print(f'  Quantization error: {l1:.2f}%')            layer_stats.append({'name': name, 'indices': indices, 'dequant': dequant, 'fp16': w_fp16})

In [ ]:
idx = 1  # Change as neededselected = layer_stats[idx]print(f'Selected: {selected["name"]}')target_indices = selected['indices'].float()target_dequant = selected['dequant']print(f'Target shape: {target_indices.shape}')print(f'Unique indices: {len(torch.unique(target_indices))}')

In [ ]:
class DiffQuantize(torch.autograd.Function):    @staticmethod    def forward(ctx, x, absmax, mode='round'):        ctx.save_for_backward(x)        x_norm = torch.clamp(x / absmax.clamp(min=1e-8), -1.0, 1.0)        distances = (x_norm.unsqueeze(-1) - NF4_LEVELS.to(x.device)).abs()        if mode == 'floor':            mask = (NF4_LEVELS <= x_norm.unsqueeze(-1))            indices = mask.sum(dim=-1).clamp(1, 16) - 1        elif mode == 'ceil':            mask = (NF4_LEVELS >= x_norm.unsqueeze(-1))            indices = 16 - mask.sum(dim=-1).clamp(1, 16)        else:  # round            indices = distances.argmin(dim=-1)        quantized = NF4_LEVELS.to(x.device)[indices] * absmax        return quantized, indices.float()    @staticmethod    def backward(ctx, grad_q, grad_i):        x, = ctx.saved_tensors        return grad_q, None, Nonedef quantize(x, absmax, mode='round'):    return DiffQuantize.apply(x, absmax, mode)print('Differentiable quantization defined')

In [ ]:
class PackedLoRA(nn.Module):    def __init__(self, in_f, out_f, rank):        super().__init__()        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.01)        self.B = nn.Parameter(torch.randn(out_f, rank) * 0.01)        self.absmax = nn.Parameter(torch.ones(out_f, 1) * 0.5)    def forward(self, shape):        W = torch.matmul(self.B, self.A)        absmax_exp = self.absmax.expand(shape)        W_q, idx = quantize(W, absmax_exp)        return W_q, idxprint('PackedLoRA defined')

In [ ]:
def train_packed_lora(target_idx, target_deq, rank, lr, epochs, mode, device):    d, k = target_idx.shape    model = PackedLoRA(k, d, rank).to(device)    opt = torch.optim.AdamW(model.parameters(), lr=lr)    best_loss, best_state = float('inf'), None    target_idx_d = target_idx.to(device)    target_deq_d = target_deq.to(device)    for epoch in range(epochs):        opt.zero_grad()        W_q, idx_pred = model(target_idx_d.shape)        loss = F.mse_loss(W_q, target_deq_d) + 0.1 * F.mse_loss(idx_pred, target_idx_d)        if not torch.isfinite(loss): return float('inf'), 0, 0        loss.backward()        opt.step()        if loss.item() < best_loss:            best_loss = loss.item()            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}    model.load_state_dict(best_state)    with torch.no_grad():        W_f, idx_f = model(target_idx_d.shape)        l1 = torch.mean(torch.abs(W_f - target_deq_d)).item() / torch.mean(torch.abs(target_deq_d)).item() * 100        acc = (idx_f.round() == target_idx_d).float().mean().item() * 100    return l1, epochs, accprint('Training function defined')

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'print('Testing quantization modes:')for mode in ['round', 'floor', 'ceil']:    print(f'\nMode: {mode}')    for rank in [4, 8, 16, 32]:        l1, ep, acc = train_packed_lora(target_indices, target_dequant, rank, 0.01, 300, mode, device)        print(f'  Rank {rank:2d}: L1={l1:5.2f}% Acc={acc:5.1f}%')

In [ ]:
BEST_MODE = 'round'def objective(trial):    rank = trial.suggest_int('rank', 2, 32, log=True)    lr = trial.suggest_float('lr', 0.001, 0.1, log=True)    epochs = trial.suggest_int('epochs', 100, 500, log=True)    l1, ep, acc = train_packed_lora(target_indices, target_dequant, rank, lr, epochs, BEST_MODE, device)    trial.set_user_attr('l1', l1)    trial.set_user_attr('acc', acc)    return l1 + (100 - acc) * 0.1import osstudy_name = f'packed_q4_{BEST_MODE}_{selected["name"].replace(".", "_")}'db_file = f'{DRIVE_BASE}/databases/{study_name}.db'os.makedirs(os.path.dirname(db_file), exist_ok=True)study = optuna.create_study(study_name=study_name, storage=f'sqlite:///{db_file}',                          direction='minimize', load_if_exists=True)study.optimize(objective, n_trials=20, show_progress_bar=True)print(f'Best: L1={study.best_trial.user_attrs["l1"]:.2f}% Acc={study.best_trial.user_attrs["acc"]:.1f}%')print(f'Rank: {study.best_params["rank"]}, LR: {study.best_params["lr"]:.4f}')